**6.1. Building a RAG system with LangChain**

**Retrieval & Generation**

In [4]:
from dotenv import load_dotenv

from langchain_chroma import Chroma
# this don't work
#from langchain_huggingface import HuggingFaceEmbeddings 

# this is for converting our query to embedding
from langchain_openai import OpenAIEmbeddings
# from langchain_groq import ChatGroq

# we use open ai llm to finally give us the response
from langchain_openai import ChatOpenAI

# langchain library has been restructured recently. Use langchain_classic for older imports as mentioned below.
# its retrival question answering which orchatrated everythin 
from langchain_classic.chains import RetrievalQA

In [5]:
# Load OPENAI_API_KEY from .env
load_dotenv()


True

In [6]:
# configuration

# just giving the path here so easy to use
vector_db_path = r"C:\Users\matule\OneDrive - Capgemini\Desktop\Training\Agentic Ai\RAG_retrievial_augmented_generation\vector_db"
collection_name  = "document_collection"

In [7]:
# # loading the embedding model - default model
# from langchain_openai import OpenAIEmbeddings


# # 2. Point to the Capgemini gateway
# embedding = OpenAIEmbeddings(
#     base_url="https://openai.generative.engine.capgemini.com/v1",
#     model="text-embedding-3-small"  # or "openai.text-embedding-3-small"
# )

import os
from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.embeddings import Embeddings

# Load API key
load_dotenv()

class CapgeminiOpenAIEmbeddings(Embeddings):
    def __init__(self, model="text-embedding-3-small"):
        self.client = OpenAI(
            base_url="https://openai.generative.engine.capgemini.com/v1"
        )
        self.model = model

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embeds a list of document chunks one by one."""
        embeddings = []
        for text in texts:
            response = self.client.embeddings.create(
                model=self.model,
                input=text
            )
            embeddings.append(response.data[0].embedding)
        return embeddings

    def embed_query(self, text: str) -> list[float]:
        """Embeds a single query string."""
        response = self.client.embeddings.create(
            model=self.model,
            input=text
        )
        return response.data[0].embedding

# Initialize the embedding object
embedding = CapgeminiOpenAIEmbeddings(model="text-embedding-3-small")


In [8]:
# Initialize OpenAI LLM pointing to Capgemini Enterprise Gateway
llm = ChatOpenAI(
    base_url="https://openai.generative.engine.capgemini.com/v1",
    model="openai.gpt-4o",  # or "openai.gpt-5-nano" / "anthropic.claude-haiku-4-5-20251001-v1:0"
    temperature=0.0
)


This is loading the vector store to retrive the data from here 

In [9]:
#loading the vector store

vector_store = Chroma(
    collection_name=collection_name, # want to use this collection from vector db
    embedding_function=embedding,
    persist_directory=vector_db_path # this is our persistes directory it will now vanish once we close our program
)

***This retriver helps to retrive the data from the store.***

In [10]:
retriever = vector_store.as_retriever()

In [11]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # how we can retrive the data, diffrent configurations
    retriever=retriever,
    return_source_documents=True # we want infromation about source document as well
)

In [12]:
query = "What does the document say about Adaptive Radiation?"
#here do create a dictionary 
#here the key should be query this should not be changed
response = qa_chain.invoke({"query":query})
print(response["result"])

print("-"*80)
for source in response["source_documents"]: #to print the source from where the data was taken
    print(source.metadata)

The document describes adaptive radiation as the process of evolution of different species in a given geographical area starting from a common ancestor and radiating to other areas or habitats. It provides the example of Darwin's finches on the Galapagos Islands, where various finch species evolved from an original seed-eating ancestor into forms with different beak shapes, allowing them to exploit different food sources like insects and plants. Another example given is the adaptive radiation of marsupials in Australia, where a variety of marsupials evolved from an ancestral stock within the Australian continent. The document also mentions that when more than one adaptive radiation occurs in an isolated geographical area, it can lead to convergent evolution, as seen with placental mammals in Australia evolving into forms similar to corresponding marsupials.
--------------------------------------------------------------------------------
{'source': 'c:\\Users\\matule\\OneDrive - Capgemi

In [13]:
response

{'query': 'What does the document say about Adaptive Radiation?',
 'result': "The document describes adaptive radiation as the process of evolution of different species in a given geographical area starting from a common ancestor and radiating to other areas or habitats. It provides the example of Darwin's finches on the Galapagos Islands, where various finch species evolved from an original seed-eating ancestor into forms with different beak shapes, allowing them to exploit different food sources like insects and plants. Another example given is the adaptive radiation of marsupials in Australia, where a variety of marsupials evolved from an ancestral stock within the Australian continent. The document also mentions that when more than one adaptive radiation occurs in an isolated geographical area, it can lead to convergent evolution, as seen with placental mammals in Australia evolving into forms similar to corresponding marsupials.",
 'source_documents': [Document(id='a0d5a974-f05c-4

In [ ]:
query = "What does the document say about Evolution and Ecosystem?"
response = qa_chain.invoke({"query":query})
print(response["result"])
print("-"*50)

for source in response["source_documents"]:
    print(source.metadata)

The document discusses the following points related to Evolution and Ecosystem:

**Evolution:**

1. The origin of life on earth is understood against the background of the origin of the universe, especially the earth.
2. Most scientists believe that chemical evolution (formation of biomolecules) preceded the appearance of the first cellular forms of life.
3. The subsequent events in the evolution of life are a conjectured story based on Darwinian ideas of organic evolution by natural selection.
4. The diversity of life forms on earth has been changing over millions of years.
5. Variations in a population result in variable fitness, and other phenomena like habitat fragmentation and genetic drift may accentuate these variations leading to the appearance of new species and hence evolution.
6. Homology is accounted for by the idea of branching descent.
7. Study of comparative anatomy, fossils, and comparative biochemistry provides evidence for evolution.
8. The story of evolution of moder